# 데이터 결합 — merge · concat · join 정답지

이 노트북은 `Pandas_2_merge_concat_Practice.ipynb`의 모든 연습문제(1~12번)에 대한 **모범 풀이 코드와 결과 해석**을 수록한 정답지입니다.
행/열 연결(concat), 조인(merge/join), 중복 키 검증(validate), 다대일/일대다 관계 및 결측 발생 원인을 단계별로 명확하게 규명합니다.


## 처음 배우는 분을 위한 안내

키(key)는 두 표에서 같은 대상을 찾는 열입니다. customer_id는 고객 번호입니다. online·offline은 한 행이 주문 한 건, master·summary는 한 행이 고객 한 명입니다. concat은 표를 이어 붙이고 merge는 같은 키로 정보를 붙입니다. 인덱스는 행 이름표, NaN은 비어 있는 값입니다. many_to_one은 여러 주문에 고객 정보 한 행씩만 연결되는지 검사하는 규칙입니다.

**학습 순서:** 문제 1~6은 필수 기초, 7~12는 선택 도전입니다. 8~10·12번은 문제 2의 all_orders를, 10번은 9번의 master_dup을 씁니다. 앞 문제를 못 풀었다면 정답의 해당 풀이를 확인한 뒤 이어가세요.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

## 실행 준비

Python 3와 pandas가 필요합니다. 정렬 노트북의 rrule에는 python-dateutil, EDA의 선택 그래프에는 matplotlib이 필요합니다. 없는 경우 별도 코드 셀에서 `%pip install pandas python-dateutil matplotlib`을 한 번 실행하세요. 이 노트북은 외부 CSV 없이 실습 데이터가 생성됩니다.

Colab에서는 `.ipynb`를 열고 준비 셀부터 실행하세요. 아래 코드는 실습 데이터 생성만 수행합니다.

In [ ]:
import pandas as pd

# 학습용 가상 데이터. 온라인·오프라인 각각 3행입니다.
online = pd.DataFrame({
    'order_id': [101, 102, 103], 'customer_id': [1, 2, 1],
    'revenue': [100, 200, 150], 'region': ['South', 'North', 'South']
})
offline = pd.DataFrame({
    'order_id': [104, 105, 106], 'customer_id': [3, 4, 2],
    'revenue': [300, 400, 250], 'region': ['East', 'West', 'North']
})
master = pd.DataFrame({
    'customer_id': [1, 2, 3, 5],
    'membership_tier': ['Gold', 'Silver', 'Bronze', 'Gold'],
    'region': ['South', 'North', 'East', 'Central']
})
summary = pd.DataFrame({
    'customer_id': [1, 2, 3, 4], 'total_orders': [2, 2, 1, 1],
    'total_spent': [250, 450, 300, 400]
})
store_a = pd.DataFrame({'product': ['A', 'B'], 'price': [100, 200]})
store_b = pd.DataFrame({'product': ['C', 'D'], 'quantity': [2, 3]})
product_info = pd.DataFrame({'product': ['A', 'B', 'C']}, index=[10, 20, 30])
sales_info = pd.DataFrame({'quantity': [3, 1, 2]}, index=[30, 10, 20])

## 문제 1. 표의 단위와 키 확인 [필수 기초]

online·offline·master·summary의 shape, 열 이름과 customer_id 중복 수를 확인하세요. 각 표의 한 행이 무엇을 의미하는지 적으세요.

**점검 기준:** 주문표는 고객 ID가 반복될 수 있습니다. 제공 master와 summary의 고객 키는 각각 유일합니다.

**차근차근:** online, offline, master, summary를 하나씩 선택해 shape와 columns를 출력합니다. customer_id.duplicated().sum()으로 중복 개수를 셉니다. 표의 행 의미를 한 문장으로 적습니다.

In [ ]:
# 문제 1 풀이
tables = {'online': online, 'offline': offline, 'master': master, 'summary': summary}

for name, t in tables.items():
    dup_count = t['customer_id'].duplicated().sum()
    print(f"[{name:7s}] shape: {t.shape}, 열 목록: {t.columns.tolist()}, customer_id 중복 수: {dup_count}건")


**결과 해석 / 확인한 점:**

1. **각 표의 행 단위(Granularity) 및 키 특성:**
   - `online` (3행×4열), `offline` (3행×4열): **주문 1건(트랜잭션 단위)**을 나타내며, `customer_id`는 동일 고객의 반복 주문으로 중복될 수 있습니다 (실제 online에는 customer_id 1이 중복 존재).
   - `master` (4행×3열), `summary` (4행×3열): **고객 1명(고객 마스터 단위)**을 나타내며, `customer_id`의 중복 수는 0건으로 유일키(Primary Key) 역할을 합니다.
2. 결합 전에 각 표가 나타내는 단위와 키의 유일성을 확인해야 의도치 않은 행 증폭(Cartesian product)이나 데이터 왜곡을 방지할 수 있습니다.


## 문제 2. 온라인·오프라인 통합 [필수 기초]

두 주문표를 행 방향으로 연결해 all_orders를 만드세요. 인덱스를 새로 부여하고 전체 매출 합계를 확인하세요.

**점검 기준:** 6행×4열, 인덱스 0~5, revenue 합계 1400입니다.

**차근차근:** pd.concat에 두 주문표를 리스트로 전달합니다. ignore_index=True로 새 순번을 붙이고 shape와 revenue.sum()을 출력합니다.

In [ ]:
# 문제 2 풀이
all_orders = pd.concat([online, offline], ignore_index=True)

display(all_orders)
print(f"\nall_orders shape    : {all_orders.shape}")
print(f"인덱스 범위         : {all_orders.index.min()} ~ {all_orders.index.max()}")
print(f"revenue 전체 합계   : {all_orders['revenue'].sum():,}원")


**결과 해석 / 확인한 점:**

1. `pd.concat([online, offline], ignore_index=True)`를 통해 온라인(3행)과 오프라인(3행) 주문이 행 방향(`axis=0`)으로 결합되어 총 **6행 × 4열**의 통합 주문표 `all_orders`가 생성되었습니다.
2. `ignore_index=True` 옵션을 적용하여 기존 0, 1, 2 인덱스가 중복되지 않고 **0부터 5까지** 고유한 새 정수 인덱스가 순차적으로 부여되었습니다.
3. 매출 합계는 `100 + 200 + 150 + 300 + 400 + 250 = 1,400원`으로 점검 기준과 정확히 일치합니다.


## 문제 3. 서로 다른 열 연결 [필수 기초]

store_a와 store_b를 행 방향으로 연결하세요. 기본 연결과 join="inner"를 비교하고 결측이 생긴 이유를 설명하세요.

**점검 기준:** 기본 결과는 4행×3열이며 price·quantity 각각 2개 결측입니다. 공통 열 연결에는 product만 남습니다.

**차근차근:** 두 표를 기본 concat으로 연결한 결과와 join="inner" 결과를 서로 다른 변수에 저장합니다. 각 결과의 열 이름과 isna().sum()을 비교합니다.

In [ ]:
# 문제 3 풀이
concat_outer = pd.concat([store_a, store_b], ignore_index=True)
concat_inner = pd.concat([store_a, store_b], join='inner', ignore_index=True)

print("=== 기본 연결 (join='outer', 열 합집합) ===")
display(concat_outer)
print("outer 결측 개수:\n", concat_outer.isna().sum())

print("\n=== inner 연결 (join='inner', 열 교집합) ===")
display(concat_inner)


**결과 해석 / 확인한 점:**

1. **기본 연결(`join='outer'`):** 열의 합집합(`product`, `price`, `quantity`)을 유지하여 **4행 × 3열**이 됩니다. `store_a`에는 `quantity`가 없고, `store_b`에는 `price`가 없기 때문에 각각 2건의 `NaN` 결측치가 발생합니다.
2. **`join='inner'` 연결:** 두 표의 공통 열인 `product` 열만 유지되어 **4행 × 1열**이 되며 결측치는 발생하지 않지만 한쪽에만 존재하는 고유 속성 열들은 모두 유실됩니다.
3. 서로 다른 스키마의 데이터를 연결할 때는 결측값 허용 여부와 필수 보존 열을 사전에 결정해야 합니다.


## 문제 4. 좌우 연결의 기준 [필수 기초]

product_info와 sales_info를 axis=1로 연결하세요. product A의 quantity를 확인하고 단순 행 순서 연결과 차이를 설명하세요.

**점검 기준:** 인덱스 10에 A와 수량 1이 연결됩니다. 정렬된 표시 순서만 보고 대응을 정하지 마세요.

**차근차근:** axis=1로 연결한 표에서 인덱스 10인 행을 loc으로 선택합니다. 인덱스 10을 두 입력 표에서 각각 찾아 확인합니다.

In [ ]:
# 문제 4 풀이
concat_col = pd.concat([product_info, sales_info], axis=1)

display(concat_col)
print(f"\n인덱스 10의 product A에 매칭된 quantity: {concat_col.loc[10, 'quantity']}")


**결과 해석 / 확인한 점:**

1. `pd.concat(..., axis=1)`은 물리적 행 순서가 아니라 **인덱스 레이블(Index Label)**을 기준으로 열을 결합합니다.
2. `sales_info`의 첫 행은 인덱스 30(quantity=3)이지만, `product_info`의 인덱스 10인 'A'는 `sales_info`의 인덱스 10에 위치한 **`quantity=1`**과 정확히 매칭되었습니다.
3. **단순 순서 연결과의 차이:** 만약 인덱스를 무시하고 단순 순서대로 옆에 붙였다면 A에 3이 붙어 심각한 판매 수량 왜곡이 발생했을 것입니다. 판다스의 축 연결은 항상 레이블 정렬이 최우선 기준입니다.


## 문제 5. inner merge [필수 기초]

summary를 왼쪽, master를 오른쪽에 놓고 customer_id로 inner 결합하세요. 남은 고객 ID와 행 수를 확인하세요.

**점검 기준:** 고객 1·2·3의 3행입니다. 두 표 모두 고객 단위이므로 이 행 수를 주문 건수라고 부르지 않습니다.

**차근차근:** 왼쪽 표는 summary입니다. on="customer_id", how="inner"를 지정하고 customer_id 목록과 len()을 출력합니다.

In [ ]:
# 문제 5 풀이
m_inner = pd.merge(summary, master, on='customer_id', how='inner')

display(m_inner)
print(f"\nm_inner shape      : {m_inner.shape}")
print(f"남은 customer_id 목록: {m_inner['customer_id'].tolist()}")


**결과 해석 / 확인한 점:**

1. `how='inner'`는 양쪽 표에 모두 존재하는 공통 `customer_id`만을 유지합니다.
2. `summary`의 고객 [1, 2, 3, 4]와 `master`의 고객 [1, 2, 3, 5] 중 교집합인 **고객 1, 2, 3의 총 3행**이 남습니다.
3. 두 표 모두 1행이 고객 1명을 나타내는 고객 단위 표이므로, 이 결과의 3행은 '주문 3건'이 아니라 **'양쪽 시스템에 모두 등록된 고객 3명'**을 의미합니다.


## 문제 6. left merge [필수 기초]

같은 두 표를 left 결합해 고객 4가 남는지 확인하세요. 회원 등급이 없는 행을 선택하세요.

**점검 기준:** 4행이며 고객 4의 등급이 결측입니다. 결측이 미매칭 때문에 생겼는지 구분합니다.

**차근차근:** 문제 5와 같은 summary·master를 사용하되 how="left"로 바꿉니다. membership_tier.isna() 조건으로 등급이 없는 행을 고릅니다.

In [ ]:
# 문제 6 풀이
m_left = pd.merge(summary, master, on='customer_id', how='left')

display(m_left)
print(f"\nm_left shape: {m_left.shape}")

no_tier = m_left[m_left['membership_tier'].isna()]
print("\n=== 회원 등급(membership_tier)이 없는 행 ===")
display(no_tier)


**결과 해석 / 확인한 점:**

1. `how='left'`는 왼쪽 표(`summary`)의 모든 고객(1, 2, 3, 4)을 보존하므로 총 **4행**이 유지됩니다.
2. 고객 4는 주문 요약(`summary`)에는 존재하지만 고객 마스터(`master`)에는 미등록되어 있어, 결합 후 `membership_tier`와 `region`이 `NaN`으로 채워집니다.
3. 이 결측치는 원본 데이터의 누락이 아니라, **조인 키 불일치(미매칭)로 인해 병합 과정에서 발생한 결측**임을 명확히 구분해야 합니다.


## 문제 7. right와 outer 비교 [선택 도전]

summary와 master를 right 및 outer로 각각 결합하세요. indicator를 사용해 고객 4와 5의 소속을 설명하세요.

**점검 기준:** right는 4행, outer는 5행입니다. outer에서 고객 4는 left_only, 고객 5는 right_only입니다.

**차근차근:** right와 outer 결과를 따로 저장합니다. outer에는 indicator=True를 줍니다. _merge 열의 both는 양쪽, left_only는 왼쪽에만, right_only는 오른쪽에만 있다는 뜻입니다.

In [ ]:
# 문제 7 풀이
m_right = pd.merge(summary, master, on='customer_id', how='right')
m_outer = pd.merge(summary, master, on='customer_id', how='outer', indicator=True)

print("=== m_right (4행) ===")
display(m_right)

print("\n=== m_outer (5행, indicator=True) ===")
display(m_outer)
print("\n_merge 소속 빈도:\n", m_outer['_merge'].value_counts())


**결과 해석 / 확인한 점:**

1. `m_right`는 오른쪽 표(`master`)의 고객 [1, 2, 3, 5]를 기준으로 **4행**이 생성되며, 주문 실적이 없는 고객 5의 `total_orders`, `total_spent`는 `NaN`이 됩니다.
2. `m_outer`는 양쪽 고객 [1, 2, 3, 4, 5] 전체를 보존하여 **5행**이 생성됩니다.
3. `indicator=True`로 확인한 소속:
   - 고객 1, 2, 3: `both` (주문도 있고 마스터에도 등록)
   - 고객 4: `left_only` (주문은 있으나 마스터에 미등록된 비회원 주문)
   - 고객 5: `right_only` (회원 가입은 했으나 아직 주문 이력이 없는 미구매 회원)


## 문제 8. 지역 열 충돌 처리 [선택 도전]

all_orders에 master를 left 결합하세요. 지역 열을 region_order와 region_customer로 구분하고 오른쪽 키의 유일성을 검증하세요.

**점검 기준:** 결과는 6행이며 고객 4 주문에 고객 정보가 없습니다. 검증 기준은 many_to_one입니다.

**차근차근:** 문제 2의 all_orders에 master를 붙입니다. suffixes=("_order","_customer")로 겹치는 열 이름 뒤에 꼬리표를 붙이고 validate="many_to_one"으로 오른쪽 고객 번호 중복을 검사합니다.

In [ ]:
# 문제 8 풀이
m_orders = pd.merge(
    all_orders, master,
    on='customer_id', how='left',
    suffixes=('_order', '_customer'),
    validate='many_to_one'
)

display(m_orders)
print(f"\nm_orders shape: {m_orders.shape}")
print("고객 4 주문의 마스터 정보 결측 여부:")
display(m_orders[m_orders['customer_id'] == 4])


**결과 해석 / 확인한 점:**

1. `all_orders`와 `master` 양쪽에 모두 존재하는 `region` 열이 `suffixes` 옵션에 의해 `region_order`(주문 발생 지역)와 `region_customer`(고객 기본 거주 지역)로 명확히 분리되었습니다.
2. `validate='many_to_one'` 검증을 통과하여, 왼쪽의 여러 주문(many)이 오른쪽 마스터의 고유한 단일 고객(one)에 주문별로 한 고객 행에 결합되도록 오른쪽 키의 유일성을 검사합니다. 이는 고객 정보 자체의 정확성을 보증하지는 않습니다.
3. 전체 6개 주문 중 마스터 미등록 고객 4의 105번 주문 1건은 `region_customer`와 `membership_tier`가 `NaN`으로 표기되었습니다.


## 문제 9. 병합으로 행이 늘어나는 이유 [선택 도전]

master의 customer_id=2인 행을 한 번 더 붙여 master_dup을 만드세요. all_orders와 left 결합하고 원래 결과와 행 수·매출 합계를 비교하세요.

**점검 기준:** 8행, revenue 합계 1850이 됩니다. 고객 2의 두 주문이 각각 한 번씩 추가됩니다. 원본 master는 바꾸지 마세요.

**차근차근:** master에서 고객 2인 행을 골라 pd.concat으로 master 뒤에 한 번 더 붙입니다. 이 복사본으로 결합한 결과의 len()과 revenue.sum()을 문제 8과 비교합니다.

In [ ]:
# 문제 9 풀이
# customer_id=2인 행을 중복 추가하여 master_dup 생성
dup_row = master[master['customer_id'] == 2]
master_dup = pd.concat([master, dup_row], ignore_index=True)

print("=== master_dup ===")
display(master_dup)

m_dup = pd.merge(all_orders, master_dup, on='customer_id', how='left')

print("=== m_dup (중복 키 병합 결과) ===")
display(m_dup)
print(f"\n정상 병합 행 수: {len(m_orders)}행, 매출 합계: {m_orders['revenue'].sum()}원")
print(f"중복 병합 행 수: {len(m_dup)}행, 매출 합계: {m_dup['revenue'].sum()}원")


**결과 해석 / 확인한 점:**

1. `master_dup`에 고객 2가 2개 행으로 존재하자, `all_orders`에 있던 고객 2의 주문 2건(102번 200원, 106번 250원)이 각각 2번씩 매칭되어 총 4행으로 복제되었습니다.
2. 그 결과 전체 행 수는 6행에서 **8행**으로 2행 늘어났고, 총 매출 합계 역시 1,400원에서 450원이 중복 가산된 **1,850원**으로 심각하게 왜곡되었습니다.
3. 이는 실무에서 마스터 테이블에 중복 키가 존재할 때 매출이나 수량이 이중 집계(Double counting)되는 대표적인 오류 원인입니다.


## 문제 10. 중복 키 오류로 잡기 [선택 도전]

master_dup으로 many_to_one 검증을 적용해 오류를 확인하세요. try/except로 오류 메시지를 출력하고 원인과 해결 방향을 적으세요.

**점검 기준:** MergeError를 예상합니다. 의미를 확인하지 않고 중복 행을 무조건 제거하는 해결책은 피하세요.

**차근차근:** try 블록 안에서 9번의 중복 표로 검증 결합을 시도합니다. except pd.errors.MergeError에서 메시지를 출력합니다. 이 오류는 예상된 학습 결과이며 전체 실행을 중단할 필요가 없습니다.

In [ ]:
# 문제 10 풀이
try:
    pd.merge(all_orders, master_dup, on='customer_id', how='left', validate='many_to_one')
except Exception as e:
    print(f"발생한 예외 타입: {type(e).__name__}")
    print(f"오류 메시지 내용: {e}")


**결과 해석 / 확인한 점:**

1. `validate='many_to_one'` 옵션에 의해 `MergeError: Merge keys are not unique in right dataset; not a many-to-one merge` 예외가 발생하여 중복 오류를 사전에 차단했습니다.
2. **원인 규명:** 오른쪽 데이터셋(`master_dup`)의 병합 키인 `customer_id`에 중복 값(고객 2가 2건)이 존재하여 1:N 조인 관계가 깨졌기 때문입니다.
3. **올바른 해결 방향:**
   - 맹목적으로 `drop_duplicates()`를 실행하여 임의의 행을 삭제하면 최신 등급이나 거주 지역 등 중요한 변경 내역이 유실될 수 있습니다.
   - 따라서 중복이 발생한 비즈니스 원인(예: 이력 관리 테이블인지, 입력 실수인지)을 확인하고, 유효 시작일(`effective_date`) 기준 최신 1건 필터링 또는 고유 마스터 정제 후 결합해야 합니다.


## 문제 11. 인덱스 기반 join [선택 도전]

master와 summary의 customer_id를 인덱스로 지정하고 left join하세요. 고객 5의 주문 정보를 확인한 뒤 동등한 merge 결과와 비교하세요.

**점검 기준:** 4명의 마스터 고객이 남습니다. 고객 5는 주문 요약이 결측입니다. 비교할 때 인덱스·열 순서를 맞춥니다.

**차근차근:** set_index("customer_id")로 고객 번호를 행 이름표로 만든 후 master 쪽에서 join(how="left")합니다. 같은 방향의 merge 결과도 고객 번호를 인덱스로 맞춘 뒤 비교합니다.

In [ ]:
# 문제 11 풀이
m_idx = master.set_index('customer_id')
s_idx = summary.set_index('customer_id')

# master 기준 left join
joined = m_idx.join(s_idx, how='left')

display(joined)
print("\n고객 5 정보:")
display(joined.loc[[5]])

# 동등한 merge와 비교
merged_equiv = pd.merge(master, summary, on='customer_id', how='left').set_index('customer_id')
print("\n동등한 merge 결과와 일치 여부:", joined.equals(merged_equiv))


**결과 해석 / 확인한 점:**

1. `join()` 메서드는 기본적으로 **인덱스(Index)**를 기준으로 결합합니다.
2. `master` 기준의 left join 결과 4명의 마스터 고객(1, 2, 3, 5)이 유지되며, 주문 요약이 없는 고객 5는 `total_orders`와 `total_spent`가 `NaN`으로 결측 처리됩니다.
3. `pd.merge()`에 `how='left'`를 적용하고 인덱스를 일치시킨 결과와 `joined.equals(merged_equiv)` 비교 시 완벽히 동일한 결과를 반환함을 검증했습니다.


## 문제 12. 결합 결과 점검 보고서 [선택 도전]

주문에 등급을 붙이는 목적에 맞는 결합을 골라 행 수·고유 주문 수·매출 합계·미매칭 건수를 보고하세요. 고객 요약표에 사용할 결합과 무엇이 다른지 설명하세요.

**점검 기준:** 가상 주문 자료는 6행·고유 주문 6개·매출 1400·미매칭 주문 1개입니다. 표의 단위와 보존할 모집단을 먼저 적으세요.

**차근차근:** 문제 8의 방식으로 주문을 보존합니다. len(), order_id.nunique(), revenue.sum(), membership_tier.isna().sum()을 차례로 기록합니다. 행 수와 주문 수를 따로 적는 이유를 설명합니다.

In [ ]:
# 문제 12 풀이
# 목적: 모든 주문을 보존하면서 고객 등급을 부착하는 분석
report_merge = pd.merge(
    all_orders, master,
    on='customer_id', how='left',
    suffixes=('_order', '_customer'),
    validate='many_to_one'
)

total_order_rows = len(report_merge)
unique_orders = report_merge['order_id'].nunique()
total_sales = report_merge['revenue'].sum()
unmatched_orders = report_merge['membership_tier'].isna().sum()

report_df = pd.DataFrame({
    '점검 항목': ['전체 행 수', '고유 주문 수', '매출 합계', '미매칭 주문 수'],
    '집계 결과': [f"{total_order_rows}행", f"{unique_orders}개", f"{total_sales:,}원", f"{unmatched_orders}건"]
})

display(report_df)


**결과 해석 / 확인한 점:**

### 1. 주문-등급 결합 점검 결과
- **선택한 조인 방식:** 주문이 누락되거나 이중 집계되지 않도록 `all_orders`를 왼쪽 표로 한 `left merge(validate='many_to_one')`를 선택했습니다.
- **모집단 보존:** 주문 원본 6건이 100% 보존(`고유 주문 6개`)되었으며, 전체 매출은 **1,400원**으로 정확히 유지되었습니다.
- **미매칭 건수:** 회원 마스터에 등록되지 않은 비회원 주문(고객 4의 105번 주문) **1건**이 미매칭으로 식별되었습니다.

### 2. 고객 요약표 목적 결합과의 차이점
- **주문 분석(이 문제):** 보존해야 할 기본 모집단이 **'주문(Transaction)'**이므로 `all_orders` 기준 left join을 적용합니다. 미매칭 건은 비회원 주문 관리 대책 수립에 활용됩니다.
- **고객 CRM 분석:** 보존해야 할 기본 모집단이 **'고객(Customer Master)'**이므로 `master` 기준 left join 또는 `outer join`을 적용합니다. 이를 통해 미구매 고객(고객 5)의 구매 유도 프로모션 대상을 도출하게 됩니다.


## 선택 도전: 주문 한 건을 추가해 결합 점검하기

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
import pandas as pd
extra_order = pd.DataFrame({'order_id':[107], 'customer_id':[5], 'revenue':[120], 'region':['Central']})
```

### 진행 순서

준비 셀의 online·offline·master를 그대로 둡니다.
1. online, offline, extra_order를 행 방향으로 붙여 orders_extra에 저장합니다.
2. customer_id를 기준으로 master의 등급을 왼쪽 결합합니다. 주문 지역과 고객 지역은 suffixes로 구분합니다.
3. 행 수, 매출 합계, 등급이 비어 있는 주문 수를 출력합니다.

**완료 기준:** 7행, 매출 합계 1520, 등급 미매칭 주문 1개. 기존 1~12번의 6행·1400 기준은 바꾸지 않습니다.

### 선택 도전 풀이

```python
orders_extra = pd.concat([online, offline, extra_order], ignore_index=True)
checked_extra = orders_extra.merge(master, on='customer_id', how='left', suffixes=('_order','_customer'), validate='many_to_one')
print(len(checked_extra), checked_extra['revenue'].sum(), checked_extra['membership_tier'].isna().sum())
assert len(checked_extra)==7 and checked_extra['revenue'].sum()==1520
assert checked_extra['membership_tier'].isna().sum()==1
```

## 제출 전 확인

- 준비 셀과 풀이를 위에서 아래로 실행했나요?
- 모든 점검 기준에 대해 결과와 이유를 남겼나요?
- 행 수·고유 개수·합계·평균의 분모를 구분했나요?
- 원본과 임시 변경본을 구분했나요?
- 결측·중복·제외 조건과 추가 확인할 사항을 기록했나요?